# Fused Attention CUDA Kernel — Colab

Builds, tests, and benchmarks fused tiled attention kernels with online streaming softmax.

**Includes:** prefill kernel, decode kernel (KV cache), LayerNorm, GELU, residual add.

**Before you start:** `Runtime > Change runtime type > T4 GPU`.

**Time:** ~5 minutes total.

## 0. Verify GPU Access

In [ ]:
import subprocess, torch

print(subprocess.run(['nvidia-smi'], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), (
    'No GPU. Runtime > Change runtime type > T4 GPU, then re-run.'
)
gpu = torch.cuda.get_device_name(0)
print(f'{gpu} | torch {torch.__version__} | CUDA {torch.version.cuda}')

r = subprocess.run(['nvcc', '--version'], capture_output=True, text=True)
print(r.stdout)
assert r.returncode == 0, 'nvcc not found'

cap = torch.cuda.get_device_capability()
ARCH = f'sm_{cap[0]}{cap[1]}'
print(f'Compute capability: {cap[0]}.{cap[1]} -> {ARCH}')

## 1. Write Source Files

In [ ]:
import os
os.makedirs('/content/fused-attention', exist_ok=True)
os.chdir('/content/fused-attention')
print(f'Working directory: {os.getcwd()}')

In [ ]:
%%writefile attention.cuh
#pragma once

#include <cuda_runtime.h>
#include <cstdio>
#include <cmath>
#include <cfloat>

#define CUDA_CHECK(call)                                                       \
    do {                                                                       \
        cudaError_t err = (call);                                              \
        if (err != cudaSuccess) {                                              \
            fprintf(stderr, "CUDA error at %s:%d: %s\n", __FILE__, __LINE__,  \
                    cudaGetErrorString(err));                                   \
            exit(1);                                                           \
        }                                                                      \
    } while (0)

#ifndef TILE_Q
#define TILE_Q 16
#endif

#ifndef TILE_KV
#define TILE_KV 16
#endif

#ifndef HEAD_DIM
#define HEAD_DIM 64
#endif

__host__ __device__ inline int cdiv(int a, int b) { return (a + b - 1) / b; }

void naive_attention_cuda(
    const float* Q, const float* K, const float* V, float* O,
    int B, int H, int N, int d, bool causal,
    float* workspace = nullptr);

void fused_attention_cuda(
    const float* Q, const float* K, const float* V, float* O,
    int B, int H, int N, int d, bool causal);

// Decode (single-query, KV cache)
void fused_attention_decode_cuda(
    const float* Q, const float* K_cache, const float* V_cache, float* O,
    int B, int H, int seq_len, int max_seq, int d);

void kv_cache_append_cuda(
    float* K_cache, float* V_cache,
    const float* K_new, const float* V_new,
    int B, int H, int pos, int max_seq, int d);

inline float* load_bin(const char* path, size_t num_floats) {
    FILE* f = fopen(path, "rb");
    if (!f) { fprintf(stderr, "Cannot open %s\n", path); exit(1); }
    float* buf = (float*)malloc(num_floats * sizeof(float));
    size_t nread = fread(buf, sizeof(float), num_floats, f);
    if (nread != num_floats) {
        fprintf(stderr, "%s: expected %zu floats, got %zu\n", path, num_floats, nread);
        exit(1);
    }
    fclose(f);
    return buf;
}

// numpy-style allclose: |a-b| <= atol + rtol * |b|
inline bool check(const float* a, const float* b, size_t n,
                  float rtol, const char* label) {
    const float atol = 1e-5f;
    float max_abs = 0, max_rel = 0;
    size_t worst_abs_i = 0;
    int num_fail = 0;
    for (size_t i = 0; i < n; i++) {
        float diff = fabsf(a[i] - b[i]);
        float tol = atol + rtol * fabsf(b[i]);
        if (diff > max_abs) { max_abs = diff; worst_abs_i = i; }
        float denom = fmaxf(fabsf(b[i]), 1e-8f);
        float rel = diff / denom;
        if (rel > max_rel) max_rel = rel;
        if (diff > tol) num_fail++;
    }
    bool pass = (num_fail == 0);
    printf("  %-25s max_abs=%.2e  max_rel=%.2e  fail=%d/%zu  [%s]\n",
           label, max_abs, max_rel, num_fail, n, pass ? "PASS" : "FAIL");
    if (!pass) {
        printf("    worst at i=%zu: got %.6f expected %.6f (diff=%.2e)\n",
               worst_abs_i, a[worst_abs_i], b[worst_abs_i], max_abs);
    }
    return pass;
}

In [ ]:
%%writefile elementwise.cuh
#pragma once
#include "attention.cuh"

void layernorm_cuda(
    const float* input, const float* weight, const float* bias,
    float* output, int N, int d, float eps = 1e-5f);

void gelu_cuda(const float* input, float* output, int n);

void residual_add_cuda(const float* a, const float* b, float* output, int n);

In [ ]:
%%writefile naive_attention.cu
#include "attention.cuh"

__global__ void naive_compute_scores(
    const float* __restrict__ Q,
    const float* __restrict__ K,
    float* __restrict__ S,
    int N, int d, bool causal)
{
    int bh = blockIdx.x;
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.z * blockDim.x + threadIdx.x;
    if (row >= N || col >= N) return;
    float scale = rsqrtf((float)d);
    if (causal && col > row) {
        S[bh * N * N + row * N + col] = -INFINITY;
        return;
    }
    const float* q_row = Q + bh * N * d + row * d;
    const float* k_col = K + bh * N * d + col * d;
    float dot = 0.0f;
    for (int i = 0; i < d; i++) dot += q_row[i] * k_col[i];
    S[bh * N * N + row * N + col] = dot * scale;
}

__global__ void naive_softmax_rows(float* __restrict__ S, int N)
{
    int bh = blockIdx.x;
    int row = blockIdx.y * blockDim.x + threadIdx.x;
    if (row >= N) return;
    float* s_row = S + bh * N * N + row * N;
    float m = -INFINITY;
    for (int j = 0; j < N; j++) m = fmaxf(m, s_row[j]);
    float sum = 0.0f;
    for (int j = 0; j < N; j++) {
        s_row[j] = expf(s_row[j] - m);
        sum += s_row[j];
    }
    float inv_sum = 1.0f / sum;
    for (int j = 0; j < N; j++) s_row[j] *= inv_sum;
}

__global__ void naive_attn_times_v(
    const float* __restrict__ S,
    const float* __restrict__ V,
    float* __restrict__ O,
    int N, int d)
{
    int bh = blockIdx.x;
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int dim = blockIdx.z * blockDim.x + threadIdx.x;
    if (row >= N || dim >= d) return;
    const float* s_row = S + bh * N * N + row * N;
    float acc = 0.0f;
    for (int j = 0; j < N; j++)
        acc += s_row[j] * V[bh * N * d + j * d + dim];
    O[bh * N * d + row * d + dim] = acc;
}

void naive_attention_cuda(
    const float* Q, const float* K, const float* V, float* O,
    int B, int H, int N, int d, bool causal,
    float* workspace)
{
    int BH = B * H;
    float* S = workspace;
    bool own_S = (S == nullptr);
    if (own_S)
        CUDA_CHECK(cudaMalloc(&S, (size_t)BH * N * N * sizeof(float)));
    {
        dim3 block(16, 16);
        dim3 grid(BH, cdiv(N, 16), cdiv(N, 16));
        naive_compute_scores<<<grid, block>>>(Q, K, S, N, d, causal);
    }
    {
        int threads = 256;
        dim3 grid(BH, cdiv(N, threads));
        naive_softmax_rows<<<grid, threads>>>(S, N);
    }
    {
        dim3 block(16, 16);
        dim3 grid(BH, cdiv(N, 16), cdiv(d, 16));
        naive_attn_times_v<<<grid, block>>>(S, V, O, N, d);
    }
    CUDA_CHECK(cudaGetLastError());
    if (own_S) {
        CUDA_CHECK(cudaDeviceSynchronize());
        CUDA_CHECK(cudaFree(S));
    }
}

In [ ]:
%%writefile fused_attention.cu
#include "attention.cuh"

__global__ void fused_attention_kernel(
    const float* __restrict__ Q,
    const float* __restrict__ K,
    const float* __restrict__ V,
    float* __restrict__ O,
    int N, int d, bool causal)
{
    const int bh = blockIdx.x;
    const int q_start = blockIdx.y * TILE_Q;
    const int ty = threadIdx.y;
    const int tx = threadIdx.x;
    const int q_row = q_start + ty;
    const float scale = rsqrtf((float)d);

    __shared__ float Q_s[TILE_Q][HEAD_DIM];
    __shared__ float K_s[TILE_KV][HEAD_DIM];
    __shared__ float V_s[TILE_KV][HEAD_DIM];
    __shared__ float S_s[TILE_Q][TILE_KV];

    float acc[HEAD_DIM / 32];
    for (int i = 0; i < HEAD_DIM / 32; i++) acc[i] = 0.0f;
    float row_max = -INFINITY;
    float row_sum = 0.0f;

    for (int dd = tx; dd < d; dd += 32) {
        if (q_row < N) Q_s[ty][dd] = Q[bh * N * d + q_row * d + dd];
        else Q_s[ty][dd] = 0.0f;
    }
    __syncthreads();

    int num_kv_tiles = cdiv(N, TILE_KV);
    for (int kv_tile = 0; kv_tile < num_kv_tiles; kv_tile++) {
        int kv_start = kv_tile * TILE_KV;
        if (causal && kv_start > q_start + TILE_Q - 1) break;

        for (int r = ty; r < TILE_KV; r += TILE_Q) {
            int kv_row = kv_start + r;
            for (int dd = tx; dd < d; dd += 32) {
                K_s[r][dd] = (kv_row < N) ? K[bh * N * d + kv_row * d + dd] : 0.0f;
            }
        }
        for (int r = ty; r < TILE_KV; r += TILE_Q) {
            int kv_row = kv_start + r;
            for (int dd = tx; dd < d; dd += 32) {
                V_s[r][dd] = (kv_row < N) ? V[bh * N * d + kv_row * d + dd] : 0.0f;
            }
        }
        __syncthreads();

        for (int j = 0; j < TILE_KV; j++) {
            int kv_col = kv_start + j;
            float dot = 0.0f;
            for (int dd = tx; dd < d; dd += 32) dot += Q_s[ty][dd] * K_s[j][dd];
            for (int offset = 16; offset > 0; offset >>= 1)
                dot += __shfl_down_sync(0xffffffff, dot, offset);
            if (tx == 0) {
                float s = dot * scale;
                if (causal && kv_col > q_row) s = -INFINITY;
                if (q_row >= N || kv_col >= N) s = -INFINITY;
                S_s[ty][j] = s;
            }
        }
        __syncthreads();

        float tile_max = -INFINITY;
        if (tx == 0) {
            for (int j = 0; j < TILE_KV; j++)
                tile_max = fmaxf(tile_max, S_s[ty][j]);
        }
        tile_max = __shfl_sync(0xffffffff, tile_max, 0);
        float m_new = fmaxf(row_max, tile_max);
        float correction = expf(row_max - m_new);

        float tile_sum = 0.0f;
        if (tx == 0) {
            for (int j = 0; j < TILE_KV; j++) {
                S_s[ty][j] = expf(S_s[ty][j] - m_new);
                tile_sum += S_s[ty][j];
            }
        }
        tile_sum = __shfl_sync(0xffffffff, tile_sum, 0);
        __syncthreads();

        row_sum = row_sum * correction + tile_sum;
        for (int i = 0; i < HEAD_DIM / 32; i++) {
            int dd = tx + i * 32;
            acc[i] *= correction;
            for (int j = 0; j < TILE_KV; j++)
                acc[i] += S_s[ty][j] * V_s[j][dd];
        }
        row_max = m_new;
        __syncthreads();
    }

    if (q_row < N) {
        float inv_sum = (row_sum > 0.0f) ? (1.0f / row_sum) : 0.0f;
        for (int i = 0; i < HEAD_DIM / 32; i++) {
            int dd = tx + i * 32;
            if (dd < d) O[bh * N * d + q_row * d + dd] = acc[i] * inv_sum;
        }
    }
}

void fused_attention_cuda(
    const float* Q, const float* K, const float* V, float* O,
    int B, int H, int N, int d, bool causal)
{
    int BH = B * H;
    dim3 block(32, TILE_Q);
    dim3 grid(BH, cdiv(N, TILE_Q));
    fused_attention_kernel<<<grid, block>>>(Q, K, V, O, N, d, causal);
    CUDA_CHECK(cudaGetLastError());
}

In [ ]:
%%writefile fused_attention_decode.cu
#include "attention.cuh"

#ifndef TILE_KV_DECODE
#define TILE_KV_DECODE 32
#endif

__global__ void fused_attention_decode_kernel(
    const float* __restrict__ Q,
    const float* __restrict__ K_cache,
    const float* __restrict__ V_cache,
    float* __restrict__ O,
    int seq_len, int max_seq, int d)
{
    const int bh = blockIdx.x;
    const int tx = threadIdx.x;
    const float scale = rsqrtf((float)d);

    float q[HEAD_DIM / 32];
    for (int i = 0; i < HEAD_DIM / 32; i++) {
        int dd = tx + i * 32;
        q[i] = (dd < d) ? Q[bh * d + dd] : 0.0f;
    }

    float row_max = -INFINITY;
    float row_sum = 0.0f;
    float acc[HEAD_DIM / 32];
    for (int i = 0; i < HEAD_DIM / 32; i++) acc[i] = 0.0f;

    __shared__ float K_s[TILE_KV_DECODE][HEAD_DIM];
    __shared__ float V_s[TILE_KV_DECODE][HEAD_DIM];

    const int num_tiles = cdiv(seq_len, TILE_KV_DECODE);

    for (int tile = 0; tile < num_tiles; tile++) {
        const int kv_start = tile * TILE_KV_DECODE;
        const int tile_len = min(TILE_KV_DECODE, seq_len - kv_start);

        for (int r = 0; r < TILE_KV_DECODE; r++) {
            int kv_row = kv_start + r;
            for (int dd = tx; dd < d; dd += 32)
                K_s[r][dd] = (kv_row < seq_len)
                    ? K_cache[(bh * max_seq + kv_row) * d + dd] : 0.0f;
        }
        for (int r = 0; r < TILE_KV_DECODE; r++) {
            int kv_row = kv_start + r;
            for (int dd = tx; dd < d; dd += 32)
                V_s[r][dd] = (kv_row < seq_len)
                    ? V_cache[(bh * max_seq + kv_row) * d + dd] : 0.0f;
        }
        __syncwarp();

        float tile_max = -INFINITY;
        float scores[TILE_KV_DECODE];

        for (int j = 0; j < tile_len; j++) {
            float dot = 0.0f;
            for (int i = 0; i < HEAD_DIM / 32; i++)
                dot += q[i] * K_s[j][tx + i * 32];
            for (int offset = 16; offset > 0; offset >>= 1)
                dot += __shfl_down_sync(0xffffffff, dot, offset);
            float s = __shfl_sync(0xffffffff, dot, 0) * scale;
            scores[j] = s;
            tile_max = fmaxf(tile_max, s);
        }

        float m_new = fmaxf(row_max, tile_max);
        float correction = expf(row_max - m_new);
        for (int i = 0; i < HEAD_DIM / 32; i++) acc[i] *= correction;
        row_sum *= correction;

        float tile_sum = 0.0f;
        for (int j = 0; j < tile_len; j++) {
            float p = expf(scores[j] - m_new);
            tile_sum += p;
            for (int i = 0; i < HEAD_DIM / 32; i++)
                acc[i] += p * V_s[j][tx + i * 32];
        }

        row_sum += tile_sum;
        row_max = m_new;
        __syncwarp();
    }

    float inv_sum = (row_sum > 0.0f) ? (1.0f / row_sum) : 0.0f;
    for (int i = 0; i < HEAD_DIM / 32; i++) {
        int dd = tx + i * 32;
        if (dd < d) O[bh * d + dd] = acc[i] * inv_sum;
    }
}

void fused_attention_decode_cuda(
    const float* Q, const float* K_cache, const float* V_cache, float* O,
    int B, int H, int seq_len, int max_seq, int d)
{
    fused_attention_decode_kernel<<<B * H, 32>>>(
        Q, K_cache, V_cache, O, seq_len, max_seq, d);
    CUDA_CHECK(cudaGetLastError());
}

__global__ void kv_cache_append_kernel(
    float* __restrict__ K_cache,
    float* __restrict__ V_cache,
    const float* __restrict__ K_new,
    const float* __restrict__ V_new,
    int pos, int max_seq, int d)
{
    int bh = blockIdx.x;
    for (int dd = threadIdx.x; dd < d; dd += blockDim.x) {
        K_cache[(bh * max_seq + pos) * d + dd] = K_new[bh * d + dd];
        V_cache[(bh * max_seq + pos) * d + dd] = V_new[bh * d + dd];
    }
}

void kv_cache_append_cuda(
    float* K_cache, float* V_cache,
    const float* K_new, const float* V_new,
    int B, int H, int pos, int max_seq, int d)
{
    kv_cache_append_kernel<<<B * H, 64>>>(
        K_cache, V_cache, K_new, V_new, pos, max_seq, d);
    CUDA_CHECK(cudaGetLastError());
}

In [ ]:
%%writefile elementwise.cu
#include "elementwise.cuh"

__global__ void layernorm_kernel(
    const float* __restrict__ input,
    const float* __restrict__ weight,
    const float* __restrict__ bias,
    float* __restrict__ output,
    int N, int d, float eps)
{
    int row = blockIdx.x;
    if (row >= N) return;
    int tx = threadIdx.x;
    const float* x = input + row * d;
    float* y = output + row * d;

    float sum = 0.0f;
    for (int i = tx; i < d; i += 32) sum += x[i];
    for (int off = 16; off > 0; off >>= 1)
        sum += __shfl_down_sync(0xffffffff, sum, off);
    float mean = __shfl_sync(0xffffffff, sum, 0) / (float)d;

    float var_sum = 0.0f;
    for (int i = tx; i < d; i += 32) {
        float diff = x[i] - mean;
        var_sum += diff * diff;
    }
    for (int off = 16; off > 0; off >>= 1)
        var_sum += __shfl_down_sync(0xffffffff, var_sum, off);
    float var = __shfl_sync(0xffffffff, var_sum, 0) / (float)d;
    float inv_std = rsqrtf(var + eps);

    for (int i = tx; i < d; i += 32)
        y[i] = (x[i] - mean) * inv_std * weight[i] + bias[i];
}

void layernorm_cuda(
    const float* input, const float* weight, const float* bias,
    float* output, int N, int d, float eps)
{
    layernorm_kernel<<<N, 32>>>(input, weight, bias, output, N, d, eps);
    CUDA_CHECK(cudaGetLastError());
}

__global__ void gelu_kernel(
    const float* __restrict__ input,
    float* __restrict__ output,
    int n)
{
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i >= n) return;
    float x = input[i];
    float inner = 0.7978845608f * (x + 0.044715f * x * x * x);
    output[i] = 0.5f * x * (1.0f + tanhf(inner));
}

void gelu_cuda(const float* input, float* output, int n)
{
    int threads = 256;
    gelu_kernel<<<cdiv(n, threads), threads>>>(input, output, n);
    CUDA_CHECK(cudaGetLastError());
}

__global__ void residual_add_kernel(
    const float* __restrict__ a,
    const float* __restrict__ b,
    float* __restrict__ output,
    int n)
{
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i >= n) return;
    output[i] = a[i] + b[i];
}

void residual_add_cuda(const float* a, const float* b, float* output, int n)
{
    int threads = 256;
    residual_add_kernel<<<cdiv(n, threads), threads>>>(a, b, output, n);
    CUDA_CHECK(cudaGetLastError());
}

In [ ]:
%%writefile main.cu
#include "attention.cuh"
#include <cstdlib>
#include <cstring>
#include <cstdio>

int run_test(const char* data_dir) {
    char path[512];
    snprintf(path, sizeof(path), "%s/meta.json", data_dir);
    FILE* mf = fopen(path, "r");
    if (!mf) { fprintf(stderr, "Cannot open %s\n", path); return 1; }
    char meta_buf[1024];
    size_t meta_len = fread(meta_buf, 1, sizeof(meta_buf) - 1, mf);
    (void)meta_len;
    meta_buf[sizeof(meta_buf) - 1] = 0;
    fclose(mf);

    auto parse_int = [&](const char* key) -> int {
        const char* p = strstr(meta_buf, key);
        if (!p) { fprintf(stderr, "Missing key: %s\n", key); exit(1); }
        p = strchr(p, ':');
        return atoi(p + 1);
    };

    int B = parse_int("\"B\"");
    int H = parse_int("\"H\"");
    int N = parse_int("\"N\"");
    int d = parse_int("\"d\"");
    bool causal = strstr(meta_buf, "\"causal\": true") != nullptr;

    printf("Test: B=%d H=%d N=%d d=%d causal=%d  [%s]\n", B, H, N, d, causal, data_dir);

    size_t qkv_size = (size_t)B * H * N * d;
    snprintf(path, sizeof(path), "%s/Q.bin", data_dir);
    float* h_Q = load_bin(path, qkv_size);
    snprintf(path, sizeof(path), "%s/K.bin", data_dir);
    float* h_K = load_bin(path, qkv_size);
    snprintf(path, sizeof(path), "%s/V.bin", data_dir);
    float* h_V = load_bin(path, qkv_size);
    snprintf(path, sizeof(path), "%s/O_ref.bin", data_dir);
    float* h_O_ref = load_bin(path, qkv_size);

    float *d_Q, *d_K, *d_V, *d_O;
    size_t bytes = qkv_size * sizeof(float);
    CUDA_CHECK(cudaMalloc(&d_Q, bytes));
    CUDA_CHECK(cudaMalloc(&d_K, bytes));
    CUDA_CHECK(cudaMalloc(&d_V, bytes));
    CUDA_CHECK(cudaMalloc(&d_O, bytes));
    CUDA_CHECK(cudaMemcpy(d_Q, h_Q, bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_K, h_K, bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_V, h_V, bytes, cudaMemcpyHostToDevice));

    float* h_O_out = (float*)malloc(bytes);
    bool all_pass = true;

    CUDA_CHECK(cudaMemset(d_O, 0, bytes));
    naive_attention_cuda(d_Q, d_K, d_V, d_O, B, H, N, d, causal);
    CUDA_CHECK(cudaMemcpy(h_O_out, d_O, bytes, cudaMemcpyDeviceToHost));
    all_pass &= check(h_O_out, h_O_ref, qkv_size, 1e-4f, "naive vs reference");

    CUDA_CHECK(cudaMemset(d_O, 0, bytes));
    fused_attention_cuda(d_Q, d_K, d_V, d_O, B, H, N, d, causal);
    CUDA_CHECK(cudaDeviceSynchronize());
    CUDA_CHECK(cudaMemcpy(h_O_out, d_O, bytes, cudaMemcpyDeviceToHost));
    all_pass &= check(h_O_out, h_O_ref, qkv_size, 1e-4f, "fused vs reference");

    free(h_Q); free(h_K); free(h_V); free(h_O_ref); free(h_O_out);
    CUDA_CHECK(cudaFree(d_Q)); CUDA_CHECK(cudaFree(d_K));
    CUDA_CHECK(cudaFree(d_V)); CUDA_CHECK(cudaFree(d_O));
    return all_pass ? 0 : 1;
}

void run_bench() {
    int B = 1, H = 8, d = 64;
    bool causal = true;
    int Ns[] = {128, 256, 512, 1024, 2048};
    int num_sizes = sizeof(Ns) / sizeof(Ns[0]);
    int warmup = 10, iters = 100;

    cudaDeviceProp prop;
    CUDA_CHECK(cudaGetDeviceProperties(&prop, 0));
    printf("GPU: %s (SM %d.%d, %d SMs)\n",
           prop.name, prop.major, prop.minor, prop.multiProcessorCount);
    printf("B=%d H=%d d=%d causal=%d  warmup=%d iters=%d\n\n",
           B, H, d, causal, warmup, iters);
    printf("%-8s  %12s  %12s  %8s\n", "N", "Naive (ms)", "Fused (ms)", "Speedup");
    printf("----------------------------------------------\n");

    for (int ni = 0; ni < num_sizes; ni++) {
        int N = Ns[ni];
        size_t qkv_size = (size_t)B * H * N * d;
        size_t bytes = qkv_size * sizeof(float);
        float *d_Q, *d_K, *d_V, *d_O;
        CUDA_CHECK(cudaMalloc(&d_Q, bytes));
        CUDA_CHECK(cudaMalloc(&d_K, bytes));
        CUDA_CHECK(cudaMalloc(&d_V, bytes));
        CUDA_CHECK(cudaMalloc(&d_O, bytes));
        float* h_tmp = (float*)malloc(bytes);
        srand(42);
        for (size_t i = 0; i < qkv_size; i++)
            h_tmp[i] = ((float)rand() / RAND_MAX - 0.5f) * 2.0f;
        CUDA_CHECK(cudaMemcpy(d_Q, h_tmp, bytes, cudaMemcpyHostToDevice));
        CUDA_CHECK(cudaMemcpy(d_K, h_tmp, bytes, cudaMemcpyHostToDevice));
        CUDA_CHECK(cudaMemcpy(d_V, h_tmp, bytes, cudaMemcpyHostToDevice));
        free(h_tmp);

        float* naive_ws;
        CUDA_CHECK(cudaMalloc(&naive_ws, (size_t)B * H * N * N * sizeof(float)));
        cudaEvent_t start, stop;
        CUDA_CHECK(cudaEventCreate(&start));
        CUDA_CHECK(cudaEventCreate(&stop));

        for (int i = 0; i < warmup; i++)
            naive_attention_cuda(d_Q, d_K, d_V, d_O, B, H, N, d, causal, naive_ws);
        CUDA_CHECK(cudaEventRecord(start));
        for (int i = 0; i < iters; i++)
            naive_attention_cuda(d_Q, d_K, d_V, d_O, B, H, N, d, causal, naive_ws);
        CUDA_CHECK(cudaEventRecord(stop));
        CUDA_CHECK(cudaEventSynchronize(stop));
        float naive_ms;
        CUDA_CHECK(cudaEventElapsedTime(&naive_ms, start, stop));
        naive_ms /= iters;

        for (int i = 0; i < warmup; i++)
            fused_attention_cuda(d_Q, d_K, d_V, d_O, B, H, N, d, causal);
        CUDA_CHECK(cudaEventRecord(start));
        for (int i = 0; i < iters; i++)
            fused_attention_cuda(d_Q, d_K, d_V, d_O, B, H, N, d, causal);
        CUDA_CHECK(cudaEventRecord(stop));
        CUDA_CHECK(cudaEventSynchronize(stop));
        float fused_ms;
        CUDA_CHECK(cudaEventElapsedTime(&fused_ms, start, stop));
        fused_ms /= iters;

        printf("%-8d  %12.4f  %12.4f  %7.2fx\n",
               N, naive_ms, fused_ms, naive_ms / fused_ms);
        CUDA_CHECK(cudaEventDestroy(start)); CUDA_CHECK(cudaEventDestroy(stop));
        CUDA_CHECK(cudaFree(naive_ws));
        CUDA_CHECK(cudaFree(d_Q)); CUDA_CHECK(cudaFree(d_K));
        CUDA_CHECK(cudaFree(d_V)); CUDA_CHECK(cudaFree(d_O));
    }
}

int main(int argc, char** argv) {
    if (argc < 2) {
        printf("Usage: %s test [dir] | bench\n", argv[0]);
        return 1;
    }
    if (strcmp(argv[1], "test") == 0) {
        const char* dir = (argc > 2) ? argv[2] : "test_data";
        int rc = run_test(dir);
        if (rc == 0) {
            int extra_ns[] = {37, 127, 200};
            for (int i = 0; i < 3; i++) {
                char subdir[256];
                snprintf(subdir, sizeof(subdir), "%s/N%d", dir, extra_ns[i]);
                char meta_path[300];
                snprintf(meta_path, sizeof(meta_path), "%s/meta.json", subdir);
                FILE* f = fopen(meta_path, "r");
                if (f) { fclose(f); rc |= run_test(subdir); }
            }
        }
        printf("\n%s\n", rc == 0 ? "ALL TESTS PASSED" : "SOME TESTS FAILED");
        return rc;
    } else if (strcmp(argv[1], "bench") == 0) {
        run_bench();
        return 0;
    }
    fprintf(stderr, "Unknown command: %s\n", argv[1]);
    return 1;
}

In [ ]:
%%writefile test_day1.cu
#include "attention.cuh"
#include "elementwise.cuh"
#include <cstdlib>
#include <cstring>
#include <cstdio>

static bool parse_meta_int(const char* buf, const char* key, int* out) {
    const char* p = strstr(buf, key);
    if (!p) return false;
    p = strchr(p, ':');
    *out = atoi(p + 1);
    return true;
}

static bool load_meta(const char* dir, char* buf, size_t bufsize) {
    char path[512];
    snprintf(path, sizeof(path), "%s/meta.json", dir);
    FILE* f = fopen(path, "r");
    if (!f) { fprintf(stderr, "Cannot open %s\n", path); return false; }
    size_t n = fread(buf, 1, bufsize - 1, f);
    (void)n;
    buf[bufsize - 1] = 0;
    fclose(f);
    return true;
}

static int test_decode(const char* dir) {
    char meta[1024];
    if (!load_meta(dir, meta, sizeof(meta))) return 1;
    int B, H, d, seq_len, max_seq;
    parse_meta_int(meta, "\"B\"", &B);
    parse_meta_int(meta, "\"H\"", &H);
    parse_meta_int(meta, "\"d\"", &d);
    parse_meta_int(meta, "\"seq_len\"", &seq_len);
    parse_meta_int(meta, "\"max_seq\"", &max_seq);
    int BH = B * H;
    printf("Decode: B=%d H=%d seq_len=%d d=%d\n", B, H, seq_len, d);

    char path[512];
    snprintf(path, sizeof(path), "%s/Q.bin", dir);
    float* h_Q = load_bin(path, (size_t)BH * d);
    snprintf(path, sizeof(path), "%s/K_cache.bin", dir);
    float* h_K = load_bin(path, (size_t)BH * seq_len * d);
    snprintf(path, sizeof(path), "%s/V_cache.bin", dir);
    float* h_V = load_bin(path, (size_t)BH * seq_len * d);
    snprintf(path, sizeof(path), "%s/O_ref.bin", dir);
    float* h_O_ref = load_bin(path, (size_t)BH * d);

    float *d_Q, *d_K, *d_V, *d_O;
    CUDA_CHECK(cudaMalloc(&d_Q, (size_t)BH * d * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&d_K, (size_t)BH * max_seq * d * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&d_V, (size_t)BH * max_seq * d * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&d_O, (size_t)BH * d * sizeof(float)));
    CUDA_CHECK(cudaMemset(d_K, 0, (size_t)BH * max_seq * d * sizeof(float)));
    CUDA_CHECK(cudaMemset(d_V, 0, (size_t)BH * max_seq * d * sizeof(float)));
    CUDA_CHECK(cudaMemcpy(d_Q, h_Q, (size_t)BH * d * sizeof(float), cudaMemcpyHostToDevice));
    for (int bh = 0; bh < BH; bh++) {
        CUDA_CHECK(cudaMemcpy(d_K + bh * max_seq * d, h_K + bh * seq_len * d,
                              (size_t)seq_len * d * sizeof(float), cudaMemcpyHostToDevice));
        CUDA_CHECK(cudaMemcpy(d_V + bh * max_seq * d, h_V + bh * seq_len * d,
                              (size_t)seq_len * d * sizeof(float), cudaMemcpyHostToDevice));
    }
    CUDA_CHECK(cudaMemset(d_O, 0, (size_t)BH * d * sizeof(float)));

    fused_attention_decode_cuda(d_Q, d_K, d_V, d_O, B, H, seq_len, max_seq, d);
    CUDA_CHECK(cudaDeviceSynchronize());

    float* h_O_out = (float*)malloc((size_t)BH * d * sizeof(float));
    CUDA_CHECK(cudaMemcpy(h_O_out, d_O, (size_t)BH * d * sizeof(float), cudaMemcpyDeviceToHost));
    bool pass = check(h_O_out, h_O_ref, (size_t)BH * d, 1e-4f, "decode vs reference");

    free(h_Q); free(h_K); free(h_V); free(h_O_ref); free(h_O_out);
    CUDA_CHECK(cudaFree(d_Q)); CUDA_CHECK(cudaFree(d_K));
    CUDA_CHECK(cudaFree(d_V)); CUDA_CHECK(cudaFree(d_O));
    return pass ? 0 : 1;
}

static int test_layernorm(const char* dir) {
    char meta[1024];
    if (!load_meta(dir, meta, sizeof(meta))) return 1;
    int N, d;
    parse_meta_int(meta, "\"N\"", &N);
    parse_meta_int(meta, "\"d\"", &d);
    printf("LayerNorm: N=%d d=%d\n", N, d);

    char path[512];
    snprintf(path, sizeof(path), "%s/input.bin", dir);
    float* h_input = load_bin(path, (size_t)N * d);
    snprintf(path, sizeof(path), "%s/weight.bin", dir);
    float* h_weight = load_bin(path, d);
    snprintf(path, sizeof(path), "%s/bias.bin", dir);
    float* h_bias = load_bin(path, d);
    snprintf(path, sizeof(path), "%s/output_ref.bin", dir);
    float* h_ref = load_bin(path, (size_t)N * d);

    float *d_input, *d_weight, *d_bias, *d_output;
    size_t bytes = (size_t)N * d * sizeof(float);
    CUDA_CHECK(cudaMalloc(&d_input, bytes));
    CUDA_CHECK(cudaMalloc(&d_weight, d * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&d_bias, d * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&d_output, bytes));
    CUDA_CHECK(cudaMemcpy(d_input, h_input, bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_weight, h_weight, d * sizeof(float), cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_bias, h_bias, d * sizeof(float), cudaMemcpyHostToDevice));

    layernorm_cuda(d_input, d_weight, d_bias, d_output, N, d);
    CUDA_CHECK(cudaDeviceSynchronize());

    float* h_out = (float*)malloc(bytes);
    CUDA_CHECK(cudaMemcpy(h_out, d_output, bytes, cudaMemcpyDeviceToHost));
    bool pass = check(h_out, h_ref, (size_t)N * d, 1e-3f, "layernorm vs reference");

    free(h_input); free(h_weight); free(h_bias); free(h_ref); free(h_out);
    CUDA_CHECK(cudaFree(d_input)); CUDA_CHECK(cudaFree(d_weight));
    CUDA_CHECK(cudaFree(d_bias)); CUDA_CHECK(cudaFree(d_output));
    return pass ? 0 : 1;
}

static int test_gelu(const char* dir) {
    char meta[1024];
    if (!load_meta(dir, meta, sizeof(meta))) return 1;
    int n;
    parse_meta_int(meta, "\"n\"", &n);
    printf("GELU: n=%d\n", n);

    char path[512];
    snprintf(path, sizeof(path), "%s/input.bin", dir);
    float* h_input = load_bin(path, n);
    snprintf(path, sizeof(path), "%s/output_ref.bin", dir);
    float* h_ref = load_bin(path, n);

    float *d_input, *d_output;
    size_t bytes = n * sizeof(float);
    CUDA_CHECK(cudaMalloc(&d_input, bytes));
    CUDA_CHECK(cudaMalloc(&d_output, bytes));
    CUDA_CHECK(cudaMemcpy(d_input, h_input, bytes, cudaMemcpyHostToDevice));

    gelu_cuda(d_input, d_output, n);
    CUDA_CHECK(cudaDeviceSynchronize());

    float* h_out = (float*)malloc(bytes);
    CUDA_CHECK(cudaMemcpy(h_out, d_output, bytes, cudaMemcpyDeviceToHost));
    bool pass = check(h_out, h_ref, n, 1e-4f, "gelu vs reference");

    free(h_input); free(h_ref); free(h_out);
    CUDA_CHECK(cudaFree(d_input)); CUDA_CHECK(cudaFree(d_output));
    return pass ? 0 : 1;
}

static int test_residual(const char* dir) {
    char meta[1024];
    if (!load_meta(dir, meta, sizeof(meta))) return 1;
    int n;
    parse_meta_int(meta, "\"n\"", &n);
    printf("Residual: n=%d\n", n);

    char path[512];
    snprintf(path, sizeof(path), "%s/a.bin", dir);
    float* h_a = load_bin(path, n);
    snprintf(path, sizeof(path), "%s/b.bin", dir);
    float* h_b = load_bin(path, n);
    snprintf(path, sizeof(path), "%s/output_ref.bin", dir);
    float* h_ref = load_bin(path, n);

    float *d_a, *d_b, *d_out;
    size_t bytes = n * sizeof(float);
    CUDA_CHECK(cudaMalloc(&d_a, bytes));
    CUDA_CHECK(cudaMalloc(&d_b, bytes));
    CUDA_CHECK(cudaMalloc(&d_out, bytes));
    CUDA_CHECK(cudaMemcpy(d_a, h_a, bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_b, h_b, bytes, cudaMemcpyHostToDevice));

    residual_add_cuda(d_a, d_b, d_out, n);
    CUDA_CHECK(cudaDeviceSynchronize());

    float* h_out = (float*)malloc(bytes);
    CUDA_CHECK(cudaMemcpy(h_out, d_out, bytes, cudaMemcpyDeviceToHost));
    bool pass = check(h_out, h_ref, n, 1e-6f, "residual vs reference");

    free(h_a); free(h_b); free(h_ref); free(h_out);
    CUDA_CHECK(cudaFree(d_a)); CUDA_CHECK(cudaFree(d_b)); CUDA_CHECK(cudaFree(d_out));
    return pass ? 0 : 1;
}

int main(int argc, char** argv) {
    if (argc < 2) { printf("Usage: %s test [dir] | bench\n", argv[0]); return 1; }
    if (strcmp(argv[1], "test") == 0) {
        const char* dir = (argc > 2) ? argv[2] : "test_day1";
        int rc = 0;
        printf("\n=== Decode Attention Tests ===\n");
        { char s[256]; snprintf(s, sizeof(s), "%s/decode_B1_H4_S128", dir); rc |= test_decode(s); }
        { char s[256]; snprintf(s, sizeof(s), "%s/decode_B1_H4_S37", dir);  rc |= test_decode(s); }
        { char s[256]; snprintf(s, sizeof(s), "%s/decode_B2_H4_S256", dir); rc |= test_decode(s); }
        printf("\n=== LayerNorm Tests ===\n");
        { char s[256]; snprintf(s, sizeof(s), "%s/layernorm_N128_d256", dir); rc |= test_layernorm(s); }
        { char s[256]; snprintf(s, sizeof(s), "%s/layernorm_N37_d256", dir);  rc |= test_layernorm(s); }
        { char s[256]; snprintf(s, sizeof(s), "%s/layernorm_N128_d64", dir);  rc |= test_layernorm(s); }
        printf("\n=== GELU Tests ===\n");
        { char s[256]; snprintf(s, sizeof(s), "%s/gelu_n32768", dir); rc |= test_gelu(s); }
        { char s[256]; snprintf(s, sizeof(s), "%s/gelu_n1000", dir);  rc |= test_gelu(s); }
        printf("\n=== Residual Add Tests ===\n");
        { char s[256]; snprintf(s, sizeof(s), "%s/residual_n32768", dir); rc |= test_residual(s); }
        printf("\n%s\n", rc == 0 ? "ALL DAY 1 TESTS PASSED" : "SOME TESTS FAILED");
        return rc;
    } else if (strcmp(argv[1], "bench") == 0) {
        int B = 1, H = 8, d = 64, max_seq = 2112;
        int warmup = 10, iters = 100;
        int seq_lens[] = {128, 256, 512, 1024, 2048};
        cudaDeviceProp prop;
        CUDA_CHECK(cudaGetDeviceProperties(&prop, 0));
        printf("Decode benchmark on %s\n", prop.name);
        printf("%-10s  %12s\n", "seq_len", "Decode (ms)");
        printf("-------------------------\n");
        int BH = B * H;
        float *d_Q, *d_K, *d_V, *d_O;
        CUDA_CHECK(cudaMalloc(&d_Q, (size_t)BH * d * sizeof(float)));
        CUDA_CHECK(cudaMalloc(&d_K, (size_t)BH * max_seq * d * sizeof(float)));
        CUDA_CHECK(cudaMalloc(&d_V, (size_t)BH * max_seq * d * sizeof(float)));
        CUDA_CHECK(cudaMalloc(&d_O, (size_t)BH * d * sizeof(float)));
        float* h_tmp = (float*)malloc((size_t)BH * max_seq * d * sizeof(float));
        srand(42);
        for (size_t i = 0; i < (size_t)BH * max_seq * d; i++)
            h_tmp[i] = ((float)rand() / RAND_MAX - 0.5f) * 2.0f;
        CUDA_CHECK(cudaMemcpy(d_K, h_tmp, (size_t)BH * max_seq * d * sizeof(float), cudaMemcpyHostToDevice));
        CUDA_CHECK(cudaMemcpy(d_V, h_tmp, (size_t)BH * max_seq * d * sizeof(float), cudaMemcpyHostToDevice));
        CUDA_CHECK(cudaMemcpy(d_Q, h_tmp, (size_t)BH * d * sizeof(float), cudaMemcpyHostToDevice));
        free(h_tmp);
        cudaEvent_t start, stop;
        CUDA_CHECK(cudaEventCreate(&start));
        CUDA_CHECK(cudaEventCreate(&stop));
        for (int si = 0; si < 5; si++) {
            int sl = seq_lens[si];
            for (int i = 0; i < warmup; i++)
                fused_attention_decode_cuda(d_Q, d_K, d_V, d_O, B, H, sl, max_seq, d);
            CUDA_CHECK(cudaEventRecord(start));
            for (int i = 0; i < iters; i++)
                fused_attention_decode_cuda(d_Q, d_K, d_V, d_O, B, H, sl, max_seq, d);
            CUDA_CHECK(cudaEventRecord(stop));
            CUDA_CHECK(cudaEventSynchronize(stop));
            float ms;
            CUDA_CHECK(cudaEventElapsedTime(&ms, start, stop));
            printf("%-10d  %12.4f\n", sl, ms / iters);
        }
        CUDA_CHECK(cudaEventDestroy(start)); CUDA_CHECK(cudaEventDestroy(stop));
        CUDA_CHECK(cudaFree(d_Q)); CUDA_CHECK(cudaFree(d_K));
        CUDA_CHECK(cudaFree(d_V)); CUDA_CHECK(cudaFree(d_O));
        return 0;
    }
    fprintf(stderr, "Unknown command: %s\n", argv[1]);
    return 1;
}

In [ ]:
%%writefile generate_reference.py
import argparse, json, os
import torch
import torch.nn.functional as F

def naive_attention(Q, K, V, causal=True):
    B, H, N, d = Q.shape
    scale = d ** -0.5
    S = (Q @ K.transpose(-2, -1)) * scale
    if causal:
        mask = torch.triu(torch.ones(N, N, device=Q.device, dtype=torch.bool), diagonal=1)
        S = S.masked_fill(mask, float('-inf'))
    return torch.softmax(S, dim=-1) @ V

def save_tensor(t, path):
    t = t.contiguous().float()
    with open(path, 'wb') as f:
        f.write(t.numpy().tobytes())

def main():
    p = argparse.ArgumentParser()
    p.add_argument('--B', type=int, default=1)
    p.add_argument('--H', type=int, default=1)
    p.add_argument('--N', type=int, default=128)
    p.add_argument('--d', type=int, default=64)
    p.add_argument('--seed', type=int, default=42)
    p.add_argument('--causal', action='store_true', default=True)
    p.add_argument('--outdir', type=str, default='test_data')
    args = p.parse_args()
    torch.manual_seed(args.seed)
    Q = torch.randn(args.B, args.H, args.N, args.d)
    K = torch.randn(args.B, args.H, args.N, args.d)
    V = torch.randn(args.B, args.H, args.N, args.d)
    O = naive_attention(Q, K, V, causal=args.causal)
    os.makedirs(args.outdir, exist_ok=True)
    save_tensor(Q, f'{args.outdir}/Q.bin')
    save_tensor(K, f'{args.outdir}/K.bin')
    save_tensor(V, f'{args.outdir}/V.bin')
    save_tensor(O, f'{args.outdir}/O_ref.bin')
    meta = {'B': args.B, 'H': args.H, 'N': args.N, 'd': args.d,
            'causal': args.causal, 'seed': args.seed, 'dtype': 'float32'}
    with open(f'{args.outdir}/meta.json', 'w') as f:
        json.dump(meta, f, indent=2)
    print(f'Saved ({args.B},{args.H},{args.N},{args.d}) to {args.outdir}/')
    for test_N in [37, 127, 200]:
        sub = f'{args.outdir}/N{test_N}'
        os.makedirs(sub, exist_ok=True)
        torch.manual_seed(args.seed)
        Qq = torch.randn(args.B, args.H, test_N, args.d)
        Kk = torch.randn(args.B, args.H, test_N, args.d)
        Vv = torch.randn(args.B, args.H, test_N, args.d)
        Oo = naive_attention(Qq, Kk, Vv, causal=args.causal)
        save_tensor(Qq, f'{sub}/Q.bin')
        save_tensor(Kk, f'{sub}/K.bin')
        save_tensor(Vv, f'{sub}/V.bin')
        save_tensor(Oo, f'{sub}/O_ref.bin')
        with open(f'{sub}/meta.json', 'w') as f:
            json.dump({**meta, 'N': test_N}, f, indent=2)
        print(f'  + N={test_N}')

if __name__ == '__main__':
    main()

In [ ]:
%%writefile generate_day1_ref.py
import argparse, json, os
import torch
import torch.nn.functional as F

def save(t, path):
    t = t.contiguous().float()
    with open(path, 'wb') as f:
        f.write(t.numpy().tobytes())

def generate_decode_ref(outdir, B, H, d, seq_len, seed):
    sub = f'{outdir}/decode_B{B}_H{H}_S{seq_len}'
    os.makedirs(sub, exist_ok=True)
    torch.manual_seed(seed)
    Q = torch.randn(B * H, d)
    K_cache = torch.randn(B * H, seq_len, d)
    V_cache = torch.randn(B * H, seq_len, d)
    scale = d ** -0.5
    scores = (Q.unsqueeze(1) @ K_cache.transpose(-2, -1)).squeeze(1) * scale
    attn = torch.softmax(scores, dim=-1)
    O_ref = (attn.unsqueeze(1) @ V_cache).squeeze(1)
    save(Q, f'{sub}/Q.bin')
    save(K_cache, f'{sub}/K_cache.bin')
    save(V_cache, f'{sub}/V_cache.bin')
    save(O_ref, f'{sub}/O_ref.bin')
    meta = {'B': B, 'H': H, 'd': d, 'seq_len': seq_len, 'max_seq': seq_len + 64, 'seed': seed}
    with open(f'{sub}/meta.json', 'w') as f:
        json.dump(meta, f, indent=2)
    print(f'  decode: B={B} H={H} seq_len={seq_len} d={d}')

def generate_layernorm_ref(outdir, N, d, seed):
    sub = f'{outdir}/layernorm_N{N}_d{d}'
    os.makedirs(sub, exist_ok=True)
    torch.manual_seed(seed)
    x = torch.randn(N, d)
    weight = torch.randn(d)
    bias = torch.randn(d)
    ln = F.layer_norm(x, (d,), weight=weight, bias=bias, eps=1e-5)
    save(x, f'{sub}/input.bin')
    save(weight, f'{sub}/weight.bin')
    save(bias, f'{sub}/bias.bin')
    save(ln, f'{sub}/output_ref.bin')
    with open(f'{sub}/meta.json', 'w') as f:
        json.dump({'N': N, 'd': d, 'eps': 1e-5, 'seed': seed}, f, indent=2)
    print(f'  layernorm: N={N} d={d}')

def generate_gelu_ref(outdir, n, seed):
    sub = f'{outdir}/gelu_n{n}'
    os.makedirs(sub, exist_ok=True)
    torch.manual_seed(seed)
    x = torch.randn(n)
    y = F.gelu(x, approximate='tanh')
    save(x, f'{sub}/input.bin')
    save(y, f'{sub}/output_ref.bin')
    with open(f'{sub}/meta.json', 'w') as f:
        json.dump({'n': n, 'seed': seed}, f, indent=2)
    print(f'  gelu: n={n}')

def generate_residual_ref(outdir, n, seed):
    sub = f'{outdir}/residual_n{n}'
    os.makedirs(sub, exist_ok=True)
    torch.manual_seed(seed)
    a = torch.randn(n)
    b = torch.randn(n)
    save(a, f'{sub}/a.bin')
    save(b, f'{sub}/b.bin')
    save(a + b, f'{sub}/output_ref.bin')
    with open(f'{sub}/meta.json', 'w') as f:
        json.dump({'n': n, 'seed': seed}, f, indent=2)
    print(f'  residual: n={n}')

def main():
    p = argparse.ArgumentParser()
    p.add_argument('--outdir', default='test_day1')
    p.add_argument('--seed', type=int, default=42)
    args = p.parse_args()
    os.makedirs(args.outdir, exist_ok=True)
    print(f'Generating Day 1 reference data to {args.outdir}/')
    generate_decode_ref(args.outdir, B=1, H=4, d=64, seq_len=128, seed=args.seed)
    generate_decode_ref(args.outdir, B=1, H=4, d=64, seq_len=37,  seed=args.seed)
    generate_decode_ref(args.outdir, B=2, H=4, d=64, seq_len=256, seed=args.seed)
    generate_layernorm_ref(args.outdir, N=128, d=256, seed=args.seed)
    generate_layernorm_ref(args.outdir, N=37,  d=256, seed=args.seed)
    generate_layernorm_ref(args.outdir, N=128, d=64,  seed=args.seed)
    generate_gelu_ref(args.outdir, n=32768, seed=args.seed)
    generate_gelu_ref(args.outdir, n=1000,  seed=args.seed)
    generate_residual_ref(args.outdir, n=32768, seed=args.seed)
    print('Done.')

if __name__ == '__main__':
    main()

## 2. Generate Reference Data

In [ ]:
!python generate_reference.py
!python generate_day1_ref.py

## 3. Build

In [ ]:
# Prefill kernel binary
!nvcc -O3 -std=c++17 --use_fast_math -arch={ARCH} \
    -DTILE_Q=16 -DTILE_KV=16 -DHEAD_DIM=64 \
    -o attention main.cu naive_attention.cu fused_attention.cu

# Day 1 binary (decode + elementwise)
!nvcc -O3 -std=c++17 --use_fast_math -arch={ARCH} \
    -DTILE_Q=16 -DTILE_KV=16 -DHEAD_DIM=64 \
    -o test_day1 test_day1.cu fused_attention_decode.cu elementwise.cu

print('Build successful')

## 4. Correctness Tests — Prefill

In [ ]:
!./attention test test_data

## 5. Correctness Tests — Day 1 (Decode + Elementwise)

In [ ]:
!./test_day1 test test_day1

## 6. Benchmark — Prefill

In [ ]:
!./attention bench

## 7. Benchmark — Decode

In [ ]:
!./test_day1 bench

## 8. Download Results

In [ ]:
!./attention bench > bench_prefill.txt 2>&1
!./test_day1 bench > bench_decode.txt 2>&1
!cat bench_prefill.txt
print()
!cat bench_decode.txt

from google.colab import files
files.download('bench_prefill.txt')
files.download('bench_decode.txt')